<a href="https://colab.research.google.com/github/Auta01/Hugging-face/blob/main/Transformers%20_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="openai-community/gpt2-medium")

In [ ]:
prompt = 'what is machine learning'
output=pipe (prompt)

In [ ]:
print(output[0]['generated_text'])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2-medium")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2-medium")

In [ ]:
model

In [ ]:
sentence = 'unsure'
input_ids = tokenizer(sentence, return_tensors='pt')['input_ids']
input_ids

tensor([[13271,   495]])

In [ ]:
tokenizer.decode(input_ids[0])

In [ ]:
word ='unbelivable'
input_ids = tokenizer(sentence, return_tensors='pt').input_ids

In [ ]:
input_ids

tensor([[13271,   495]])

In [ ]:
for token_id in input_ids[0]:
   print(tokenizer.decode(token_id))

In [ ]:
word = 'homoscedasticity'
my_ids = tokenizer(word,return_tensors='pt').input_ids
my_ids

tensor([[26452,   418,   771,  3477,   414]])

In [ ]:
#Gereate
from transformers import AutoModelForCausalLM

In [ ]:
gpt2 = AutoModelForCausalLM.from_pretrained('gpt2')

In [ ]:
sentence =' i like machine learning to be a lot more powerful than'
token_ids = tokenizer(sentence, return_tensors='pt').input_ids

outputs=gpt2(token_ids).logits[0,-1]
tokenizer.decode(outputs.argmax())

' machine'

In [ ]:
def greedy_decode(logits):
  return torch.argmax(logits, dim=-1)

  greedy_decode(outputs)

In [ ]:
#Top k sampling
def top_k_sampling(logits, k=50):
  values, indices = torch.topk(logits, k)
  probs = F.softmax(values, dim=-1)
  sampled = torch.multinomial(probs,1)
  return indices[sampled]

  #Top -p(neucles sampling)
def top_p_sampling(logits, p=0.9):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = sorted_probs.cumsum(dim=-1)

    #mask token outside nuclues
    mask = cumulative_probs > p
    # Ensure at least one token is not masked to avoid RuntimeError in torch.multinomial
    if mask.all():
        mask[0] = False
    sorted_logits[mask] = float('-inf')
    #sample from filtered logits
    filtered_probs = F.softmax(sorted_logits, dim=-1)
    sampled = torch.multinomial(filtered_probs, 1)
    #Return token index in original vocabulary
    return sorted_indices[sampled]

##Temperature sampling
def temperature_sampling(logits, temperature=1.0):
  '''scale logits by temperature before sampling
  lower temperature leads to sharper distribution
  higher temperature leads to more uniform distribution'''

  scaled = logits/ temperature
  probs = F.softmax(scaled, dim=-1)
  return torch.multinomial(probs, num_samples=1)

def random_sampling(logits):
  'sample directly from softmax distribution without ffiltering'
  probs = F.softmax(logits, dim=-1)
  return torch.multinomial(probs,1)
sentence = 'today i decided to go to the store and buy some food'
inputs = tokenizer(sentence, return_tensors='pt')
output = gpt2(**inputs)
logits = output.logits[0,-1]

print(f'Greedy Decode:', tokenizer.decode([greedy_decode(logits)]))
print(f'Top_k_sampling:', tokenizer.decode([top_k_sampling(logits, k=10)]))
print(f'Top_p_sampling:', tokenizer.decode([top_p_sampling(logits, p=0.9)]))
print(f'Temp:', tokenizer.decode([temperature_sampling(logits,temperature=0.7)]))
print(f'Random:', tokenizer.decode([random_sampling(logits)]))

In [ ]:
tokenizer.decode(top_k_sampling(outputs))

In [ ]:
top_p_sampling(outputs, p=0.9)

In [ ]:
tokenizer.decode(top_p_sampling(outputs,p=0.9))

In [ ]:
tokenizer.decode(temperature_sampling(outputs, temperature =0.9))

In [ ]:
tokenizer.decode(random_sampling(outputs))